# Task 4 — Qwen3-ASR 1.7B Arabic pilot

Qwen3-ASR officially lists Arabic as a supported language, but does not list Bengali, Malayalam, or Urdu. This notebook therefore evaluates Arabic only; the other language/model combinations are marked unsupported and are not scored. Use a fresh Colab T4 runtime to avoid conflicts with the existing WhisperX environment. This produces a draft, not a verified transcript. Accuracy scoring requires a fluent-reviewed regional reference transcript.

Model card: https://huggingface.co/Qwen/Qwen3-ASR-1.7B (Apache-2.0). Official language list and setup guide: https://github.com/QwenLM/Qwen3-ASR


In [ ]:
# Run once in a fresh Colab GPU runtime. If Colab asks for a runtime restart, restart,
# reconnect to GPU, then continue with the next cell.
%pip -q install -U qwen-asr


In [ ]:
import gc, json, time, importlib.metadata
from pathlib import Path
from google.colab import files
import pandas as pd
import torch
from qwen_asr import Qwen3ASRModel

assert torch.cuda.is_available(), 'Set Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('qwen-asr:', importlib.metadata.version('qwen-asr'))

uploaded = files.upload()
audio_path = Path('arabic.mp3')
assert audio_path.exists(), 'Upload the project arabic.mp3 file.'

torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
load_start = time.perf_counter()
model = Qwen3ASRModel.from_pretrained(
    'Qwen/Qwen3-ASR-1.7B',
    dtype=torch.float16,  # T4-compatible; use fp16, not bfloat16
    device_map='cuda:0',
    max_inference_batch_size=1,
    max_new_tokens=512,
)
load_seconds = time.perf_counter() - load_start

torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
result = model.transcribe(audio=str(audio_path), language='Arabic')
inference_seconds = time.perf_counter() - start
peak_gib = torch.cuda.max_memory_allocated() / 1024**3
text = result[0].text
detected_language = getattr(result[0], 'language', 'Arabic')
print('Language:', detected_language)
print('Inference seconds:', round(inference_seconds, 2))
print('Peak GPU allocation (GiB):', round(peak_gib, 2))
print('Draft transcript:\n', text)

payload = {
    'model': 'Qwen/Qwen3-ASR-1.7B',
    'qwen_asr_package_version': importlib.metadata.version('qwen-asr'),
    'hardware': torch.cuda.get_device_name(0),
    'input_file': audio_path.name,
    'language': detected_language,
    'model_load_seconds': round(load_seconds, 2),
    'inference_seconds': round(inference_seconds, 2),
    'peak_allocated_gpu_gib': round(peak_gib, 2),
    'transcript_status': 'draft_unverified',
    'transcript': text,
}
json_path = Path('qwen3_asr_arabic_result.json')
csv_path = Path('qwen3_asr_arabic_result.csv')
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame([payload]).drop(columns=['transcript']).assign(transcript=text).to_csv(csv_path, index=False, encoding='utf-8-sig')
print('Saved:', json_path, csv_path)
files.download(str(json_path)); files.download(str(csv_path))


## Evaluation boundary

Do not treat the transcript as clinically correct until reviewed against the Arabic source script by a fluent reviewer. Once reference text is available, compare WER/CER, the named medical entities, and numerical values against the reference. Qwen3-ASR is unsupported for the supplied Bengali, Malayalam, and Urdu clips according to its published language list, so no zero scores or proxy scores should be assigned for them.
